# Report

### Question 1

The dataset has 2169 samples with two columns (`text` and `label`), is clean and balanced with no missing values, and contains approximately 200-230 samples per class. Text samples are relatively short, averaging 10-20 words, though some reach up to 78 words. A UMAP visualisation using Sentence-BERT embeddings shows well-separated clusters with minor overlap, suggesting the classes are reasonably distinguishable in the embedding space.

Several classification approaches could be considered. At the simpler end, a TF-IDF bag-of-words model with logistic regression or a linear SVM provides a fast, interpretable baseline that is often competitive on short text. Extending this with bigrams or trigrams can capture short phrases at no additional computational cost. A middle-ground option would be to use static word embeddings (GloVe or fastText) with mean pooling, which offers richer semantics than TF-IDF without the overhead of a transformer. At the more expensive end, fine-tuning Sentence-BERT would likely maximise accuracy but risks overfitting given the small dataset size and is computationally costly relative to the potential gain. Alternatively, zero-shot or few-shot prompting with a large LLM requires no training at all and could serve as a useful reference, though it is less reproducible and harder to deploy.
Given the evidence from the UMAP visualisation that the embedding space already separates classes well, the most practical approach is to use frozen Sentence-BERT embeddings (`all-mpnet-base-v2`) with lightweight classifiers such as logistic regression, a linear SVM, and an MLP. This strikes a good balance between representational richness and simplicity, avoids the overfitting risk of fine-tuning on a small dataset, and is fast to iterate.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import seaborn as sns

SHORT_LABEL = {
    0: "CardPayFee",
    1: "DirectDebitUnknown",
    2: "BalNotUpdCheque",
    3: "WrongCashAmt",
    4: "CashWithdrawCharge",
    5: "ChargedTwice",
    6: "DeclinedWithdraw",
    7: "TransferFee",
    8: "TransferNotRecvd",
    9: "BalNotUpdTransfer",
}

df = pd.read_csv("ds_task_dataset.csv")

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(
    df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Embedding matrix: {embeddings.shape}")

metric = "cosine" 

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.75,
    n_components=2,
    metric=metric,
    random_state=42,
)
proj = reducer.fit_transform(embeddings)
print(f"UMAP projection: {proj.shape}")

palette = sns.color_palette("tab10", 10)

fig, ax = plt.subplots(figsize=(11, 8))

for label_id in range(10):
    mask = df["label"].values == label_id
    ax.scatter(
        proj[mask, 0],
        proj[mask, 1],
        c=[palette[label_id]],
        label=SHORT_LABEL[label_id],
        s=18,
        alpha=0.7,
        linewidths=0,
    )

ax.set_title(
    f"UMAP Projection of SBERT Embeddings (all-mpnet-base-v2)\n"
    f"n_neighbors=15 · min_dist=0.75 · {metric} metric",
    fontsize=11,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(
    title="Class",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=8,
    title_fontsize=9,
    markerscale=1.8,
)
plt.tight_layout()
plt.show()

### Question 2 and 3

See `sentence_bert_analysis.ipynb` for training and results. Use `requirements.txt` to set up the environment.

### Question 4

The Sentence-BERT embedding approach combined with simple classifiers achieved strong performance on this 10-class text classification task. After hyperparameter tuning via 5-fold stratified cross-validation, Logistic Regression emerged as the best model with a test set macro F1 of 0.957 and ROC-AUC of 0.999, closely followed by Linear SVM (F1 0.954) and MLP (F1 0.954). Out of 326 test samples, only 14 were misclassified. Examining the failure cases reveals that most confusion occurs between semantically similar classes - for example, classes 0 and 7 (both relating to unexpected fees/charges), and classes 8 and 9 (both relating to transfer status and timing).

These are cases where even a human might need additional context to distinguish the correct intent. Given this level of performance, the recommendation to the business is to deploy the Logistic Regression model as the primary classifier for routing customer inquiries. However, there is little difference between Logistic Regression, MLP, or SVM in terms of performance. For the small fraction of ambiguous cases, particularly those involving fee-related or transfer-related queries, the system could flag low-confidence predictions for human review rather than auto-routing, ensuring customer satisfaction is not compromised.

### Question 5

Several avenues could further improve model performance. First, the most impactful approach would be to fine-tune the Sentence-BERT model itself on the domain-specific dataset rather than using it as a frozen feature extractor; this would allow the embeddings to better capture the subtle distinctions between semantically overlapping classes like fees vs. charges or transfer status vs. transfer timing. However, this would require further computational resources and a larger dataset to avoid overfitting.

Second, data augmentation could help - paraphrasing existing examples or generating synthetic samples for the most confused class pairs would give the model more signal on the decision boundaries that matter most.

Third, the label taxonomy itself could be revisited with domain experts: some of the misclassifications suggest that certain classes may have overlapping intents, and either merging ambiguous classes or providing clearer annotation guidelines could reduce noise in the training data.

Fourth, an ensemble or hierarchical classification approach could be explored, where a first-stage model groups related intents (e.g., all fee-related queries together) and a second-stage model distinguishes within those groups using more fine-grained features.

Finally, incorporating confidence calibration and an abstention mechanism - where the model defers uncertain predictions to a human agent - would allow the system to handle edge cases while maintaining high precision on the predictions it does make.